In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
import clip
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, random_split
from PIL import Image
import pandas as pd
import numpy as np
import os
import string

In [38]:
# 設置設備
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}")

clip_model, preprocess = clip.load("ViT-B/16", device=device)

for param in clip_model.parameters():
    param.requires_grad = False

print(clip_model.visual)

Using cuda
VisionTransformer(
  (conv1): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16), bias=False)
  (ln_pre): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (transformer): Transformer(
    (resblocks): Sequential(
      (0): ResidualAttentionBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): Sequential(
          (c_fc): Linear(in_features=768, out_features=3072, bias=True)
          (gelu): QuickGELU()
          (c_proj): Linear(in_features=3072, out_features=768, bias=True)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      )
      (1): ResidualAttentionBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (ln_1): LayerNorm((768,), eps=1e-05, 

In [39]:
class CLIP_EMNIST_Model(nn.Module):
    def __init__(self, clip_model, num_classes=62):
        super(CLIP_EMNIST_Model, self).__init__()
        self.clip_visual = clip_model.visual  # CLIP 影像 encoder
        self.clip_text = clip_model.encode_text  # CLIP 文字 encoder
        self.fc = nn.Sequential(
            nn.Linear(512 * 2, 256),  # 影像和文字特徵拼接
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)  # 分類 62 個類別
        )
        self.labels_dict = list(string.ascii_uppercase) + list(string.ascii_lowercase) + [str(i) for i in range(10)]  # 用於文本轉換

    def forward(self, images, text_labels):
        with torch.no_grad():  # 凍結 CLIP
            image_features = self.clip_visual(images)
            
            # 將文本標籤轉換為 CLIP 可處理的文本格式
            text_inputs = [f"A picture of the letter {label}" for label in text_labels]
            text_features = self.clip_text(text_inputs)

        # 將影像和文本特徵拼接
        combined_features = torch.cat((image_features, text_features), dim=-1)
        return self.fc(combined_features)



In [40]:
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # CLIP 需要 RGB
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])


test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),  # 測試集相同處理
    transforms.Resize((224, 224), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [41]:
# class EMNISTDataset(Dataset):
#     def __init__(self, npz_path, transform=None, has_labels=True):
#         # 讀取 npz 檔案
#         data = np.load(npz_path)
        
#         # 提取影像並去掉最後一個通道維度
#         self.images = data["training_images" if has_labels else "testing_images"]  
#         #self.images = self.images.squeeze(-1).astype(np.uint8)   
#                 # 根據影像維度進行調整
#         if len(self.images.shape) > 3:
#             # 如果有額外的維度 (例如通道維度)，進行 squeeze
#             self.images = self.images.squeeze(-1)
        
#         # 檢查數據範圍並歸一化
#         if self.images.max() <= 1.0 and self.images.min() >= 0:
#             # 如果數據範圍是 [0, 1]，轉換為 [0, 255]
#             self.images = (self.images * 255).astype(np.uint8)
#         elif self.images.max() > 1.0 and self.images.max() <= 255:
#             # 如果數據已在 [0, 255] 範圍內，確保類型為 uint8
#             self.images = self.images.astype(np.uint8)
#         else:
#             # 如果數據範圍異常，進行最小最大值歸一化
#             self.images = ((self.images - self.images.min()) / 
#                          (self.images.max() - self.images.min() + 1e-8) * 255).astype(np.uint8)
        

#         # 提取標籤（如果有）
#         self.labels = data["training_labels"] if has_labels else None
#         self.transform = transform
#         self.has_labels = has_labels

#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         # 轉換為 PIL 影像
#         image = Image.fromarray(self.images[idx])

#         if self.transform:
#             image = self.transform(image)

#         if self.has_labels:
#             label = int(self.labels[idx].item())  # 轉換為純量
#             return image, label
        
#         return image,  # 注意這裡要回傳 tuple

In [42]:
import string

class EMNISTDataset(Dataset):
    def __init__(self, npz_path, transform=None, has_labels=True):
        # 讀取 npz 檔案
        data = np.load(npz_path)
        
        # 提取影像並去掉最後一個通道維度
        self.images = data["training_images" if has_labels else "testing_images"]
        
        # 根據影像維度進行調整
        if len(self.images.shape) > 3:
            self.images = self.images.squeeze(-1)

        # 檢查數據範圍並歸一化
        if self.images.max() <= 1.0 and self.images.min() >= 0:
            self.images = (self.images * 255).astype(np.uint8)
        elif self.images.max() > 1.0 and self.images.max() <= 255:
            self.images = self.images.astype(np.uint8)
        else:
            self.images = ((self.images - self.images.min()) / 
                         (self.images.max() - self.images.min() + 1e-8) * 255).astype(np.uint8)
        
        # 創建標籤對應字典：0-25 對應 A-Z，26-35 對應 0-9
        self.labels_dict = list(string.ascii_uppercase) + list(string.ascii_lowercase) + [str(i) for i in range(10)]
        
        # 提取標籤（如果有）
        self.labels = data["training_labels"] if has_labels else None
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # 轉換為 PIL 影像
        image = Image.fromarray(self.images[idx])

        # 如果有 transform，應用 transform
        if self.transform:
            image = self.transform(image)

        if self.has_labels:
            label = int(self.labels[idx].item())  # 轉換為純量
            text_label = self.labels_dict[label]  # 將標籤轉換為文本
            return image, text_label
        
        return image,  # 返回影像和對應的文本標籤

In [45]:
from tqdm.notebook import tqdm  # 使用 Jupyter Notebook 版本的 tqdm
import matplotlib.pyplot as plt

# 讀取數據
train_dataset = EMNISTDataset("ntutemnist_data/emnist-byclass-train.npz", transform=train_transform)
test_dataset = EMNISTDataset("ntutemnist_data/emnist-byclass-test.npz", transform=test_transform, has_labels=False)

# 切割 10% 訓練集作為驗證集
val_size = int(0.3 * len(train_dataset))
train_size = len(train_dataset) - val_size
train_data, val_data = random_split(train_dataset, [train_size, val_size])

# DataLoader
train_loader = DataLoader(train_data, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

# 初始化模型
model = CLIP_EMNIST_Model(clip_model).to(torch.float32).to(device)
print(model.labels_dict)
print(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)

# 訓練函數
def train(model, train_loader, val_loader, epochs, save_path="model.pth"):
    best_val_acc = 0.0  # 用來追蹤最佳驗證準確率
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for images, text_labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=True, dynamic_ncols=True):
            images = images.to(device)
            
            # 將文本標籤轉換為數字標籤
            labels = torch.tensor([model.labels_dict.index(label) for label in text_labels]).to(device)

            optimizer.zero_grad()
            outputs = model(images, text_labels)  # 傳入影像和文本標籤
            loss = criterion(outputs, labels)  # 計算損失，使用數字標籤
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

        # 計算訓練與驗證準確度
        train_acc = correct / total
        val_acc = evaluate(model, val_loader)

        print(f"Epoch {epoch+1}/{epochs}: Loss={total_loss/len(train_loader):.4f}, Train Acc={train_acc:.4f}, Val Acc={val_acc:.4f}")

        # 如果目前的模型比之前的最佳驗證準確率更高，就儲存權重
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f"🔽 New best model saved at epoch {epoch+1} with Val Acc={val_acc:.4f}")

def evaluate(model, val_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, text_labels in val_loader:
            images = images.to(device)
            
            # 將文本標籤轉換為數字標籤
            labels = torch.tensor([model.labels_dict.index(label) for label in text_labels]).to(device)

            outputs = model(images, text_labels)  # 傳入影像和文本標籤
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return correct / total


# 開始訓練
# 訓練模型並儲存權重
train(model, train_loader, val_loader, epochs=50, save_path="best_model.pth")


['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
cuda


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html